# Cost, speed and quality: choosing a version of the tutor

*For learning. Nothing here runs in CI, and nothing costs money. Some numbers are measured from the real
tutor, and the rest come from a **practice recording** (section 3). Every table says which it is.*

## What's going on, in one paragraph

The tutor in "Ask about this lesson" could run several ways. It could use a big model or a small one,
send its prompt as it does today or in a form the model can cache, or show a **prepared answer** written
ahead of time. Each version is a different mix of **quality** (are the answers good?), **speed** (how
long do learners wait?) and **cost** (what does each answer cost?). This notebook is the experiment that
picks one, the way a data scientist would. You write the rule first, measure all three scores on the
same questions, keep only the versions nothing beats, and then let the rule choose.

## Words you'll meet

| Word | Plain meaning |
|---|---|
| **Version** | One way of running the tutor: a model, a prompt layout, or prepared answers |
| **Token** | A piece of text, roughly four characters, that models read and write. Prices are per million tokens |
| **Prompt cache** | The model provider keeps the start of a prompt it has just read. If the next prompt starts the same way, that part is read from the cache at about a tenth of the price |
| **Cache hit rate** | The share of input tokens read from the cache |
| **Time to first words** | How long until the answer starts to appear. Often called *time to first token* (TTFT) |
| **p95** | The time that 95 out of 100 answers arrive within: the slow case learners notice |
| **Non-inferiority** | Showing a new version is *no worse* than the old one by more than an agreed margin |
| **Pareto front** | The versions no other version beats on every score at once |

In [ ]:
import math
import random
from collections import Counter

import ipytest
import matplotlib.pyplot as plt
import pandas as pd
import pytest

from drill import tradeoff as t
from drill import tutor_prepared as tp
from drill import tutor_prepared_eval as tpe
from drill.content.build import build_data

ipytest.autoconfig(clean="test_*")

DATA = build_data()          # every lesson, as the page sees it
CHIPS = tp.chips(DATA)       # the standard questions, as the page asks them
ANSWERS = tp.load()          # the prepared answers, each with its judge's verdict
SPLITS = tpe.load_splits()   # each lesson's frozen split: dev, test or holdout

# The questions this experiment uses: every standard question on a dev lesson that has a prepared answer.
QUESTIONS = [{"lesson_id": lid, "key": c["key"], "chip": c, "prepared": ANSWERS[lid][c["key"]]}
             for lid, items in CHIPS.items() if SPLITS.get(lid) == "dev"
             for c in items if ANSWERS.get(lid, {}).get(c["key"], {}).get("q") == c["q"]]
print(f"{len(QUESTIONS)} dev questions across {len({q['lesson_id'] for q in QUESTIONS})} lessons")

**What that cell did.** It loaded every lesson, the standard questions ("When does this stop working?",
"Why not B?"...), the prepared answers and the frozen splits. Then it kept the **dev** questions, the
practice set you may look at as often as you like. Test and holdout stay untouched until the end
(section 10).

One limit to say out loud: these are the **standard** questions only. Learners also type their own, and
those can't be prepared ahead. A full experiment needs a second set of real typed questions, where the
prepared version isn't an option.

## 0. Write the plan before looking at results

> **In plain words.** Three scores never agree on a winner by themselves: the best answers usually cost
> the most. So decide *before* the numbers how you'll trade them off. Here the rule goes in an order:
> quality must hold, then speed must be good enough, then the cheapest wins (lesson "Write the plan
> before the runs").

In [ ]:
PLAN = {
    "question": "Which version of the tutor should answer the standard questions?",
    "versions": ["prepared, else Sonnet", "Opus", "Sonnet", "Sonnet, cacheable prompt", "Haiku", "Haiku, cacheable prompt", "a local model"],
    "baseline": "Sonnet with today's prompt: a stand-in for the live tutor (set it to the model your app really uses)",
    "quality": "pass rate under the same judge, on the same dev questions",
    "speed": "p95 time to first words",
    "cost": "dollars per 1,000 answers, with the prompt cache counted",
    "decision_rule": "1) quality: the lower end of the paired difference with the baseline is above -3 points; "
                     "2) speed: p95 time to first words is at most 5 seconds; 3) cost: the cheapest version left wins",
    "splits": "explore on dev; confirm the winner once on test; holdout stays for a bigger decision",
}
MARGIN, P95_LIMIT = 0.03, 5.0
for k, v in PLAN.items():
    print(f"{k:14} {v}")

## 1. What one answer costs

> **In plain words.** A model charges for what it **reads** (input tokens) and what it **writes** (output
> tokens), at different prices per million. Writing usually costs about five times more than reading
> (lesson "Tokens and cost"). The prompt cache adds two more kinds of input. A **cache read** is the start
> of the prompt read from the cache, at a tenth of the input price. A **cache write** is the first time
> that start is stored, at a quarter more than normal.

Here is one real Ask request: today's prompt for one question, and the prepared answer to it, measured in
tokens (roughly, at four characters a token).

In [ ]:
one = QUESTIONS[0]
prompt = t.today_messages(DATA, one["lesson_id"], one["chip"])[0]["content"]
answer = one["prepared"]["a"]
print(f"lesson {one['lesson_id']}, question: {one['chip']['q']}")
print(f"prompt about {t.estimate_tokens(prompt):,} tokens, answer about {t.estimate_tokens(answer):,} tokens")
print("prices per million tokens (input, output):", {m.removeprefix('databricks-claude-'): p for m, p in tp.PRICES.items()})

**Exercise.** Write `answer_cost`. `input_tokens` is the normal input, the part that was neither read
from nor written to the cache. That's how Claude's usage block counts it. Look up the model's prices in
`tp.PRICES` (a model that isn't there, like a local one, costs 0). Then add the four parts and divide by
a million:

- `input_tokens` at the input price;
- `cache_read` at the input price times `t.CACHE_READ` (0.1);
- `cache_write` at the input price times `t.CACHE_WRITE` (1.25);
- `output_tokens` at the output price.

In [ ]:
def answer_cost(model, input_tokens, output_tokens, cache_read=0, cache_write=0):
    """Dollars for one answer, with the prompt cache counted."""
    # TODO: pin, pout = tp.PRICES.get(model, (0.0, 0.0)); add the four parts; divide by 1e6
    raise NotImplementedError

In [ ]:
%%ipytest -qq


def test_plain_answer():
    assert answer_cost("databricks-claude-sonnet-5-5", 2000, 300) == pytest.approx((2000 * 2 + 300 * 10) / 1e6)


def test_cache_read_is_a_tenth():
    assert answer_cost("databricks-claude-sonnet-5-5", 200, 300, cache_read=1800) == pytest.approx((200 * 2 + 180 * 2 + 300 * 10) / 1e6)


def test_first_write_costs_more_than_no_cache():
    assert answer_cost("databricks-claude-sonnet-5-5", 200, 300, cache_write=1800) > answer_cost("databricks-claude-sonnet-5-5", 2000, 300)


def test_a_local_model_costs_nothing():
    assert answer_cost("a-local-model", 2000, 300) == 0.0

In [ ]:
for model in ("databricks-claude-opus-5-5", "databricks-claude-sonnet-5-5", "databricks-claude-haiku-5-5"):
    each = answer_cost(model, t.estimate_tokens(prompt), t.estimate_tokens(answer))
    print(f"{model.removeprefix('databricks-claude-'):10} ${each:.4f} an answer, ${1000 * each:.2f} for 1,000")

**What to look for.** The output is a smaller share of the tokens but a big share of the bill, because
writing costs five times more than reading. A version that writes shorter answers saves money even on the
same model. That's one reason the experiment measures real answer lengths, not a guess.

## 2. Could the prompt cache hit at all? (measured)

> **In plain words.** The cache only helps if the **start** of the prompt is the same from one request to
> the next, and long enough. Providers won't cache a start shorter than a minimum, often about 1,024
> tokens. It differs by model, so check the provider's prompt-caching page for the one you test. The
> request must also mark where the cacheable part ends. And the cache forgets after a few minutes
> without use.

The tutor's prompt starts with the same text for every question on a lesson: the tutor's instructions,
then the lesson. The question comes next, then the rules. So the start that **could** be cached is the
instructions plus the lesson. The **cacheable** version (`t.cacheable_messages`) moves the rules up so
they join the fixed start, and marks it for the cache. The model sees the same words in a different order.

In [ ]:
starts = pd.DataFrame([{"lesson_id": lid, "today": t.fixed_start(DATA, lid, items[0], "today"),
                        "cacheable": t.fixed_start(DATA, lid, items[0], "cacheable")}
                       for lid, items in CHIPS.items()])
print(f"fixed start, in tokens, across {len(starts)} lessons (measured from the real prompts):")
print(starts[["today", "cacheable"]].describe(percentiles=[.5, .9]).loc[["min", "50%", "90%", "max"]].round(0))
for col in ("today", "cacheable"):
    print(f"{col:9}: {(starts[col] >= t.MIN_CACHEABLE).sum()} of {len(starts)} lessons reach {t.MIN_CACHEABLE:,} tokens")
print("\nand today's request marks nothing for the cache, so today nothing is cached at all")

**What to look for.** This is a real finding about the tutor. **Today nothing is cached**: the request
marks nothing for the cache, and only a handful of lessons have a fixed start past the minimum anyway.
Moving the rules up brings about a third of the lessons past it. The rest still fall short, so the
cache can help some lessons but not all. Caching pays most when there is a long fixed part, like a long
system prompt, many tool definitions or a whole document, and the tutor's prompts are short. Section 6
shows what that's worth in dollars, next to the other levers.

These counts use the four-characters rule. A real run reads the exact numbers from the model's usage
block (`cache_read_input_tokens` and `cache_creation_input_tokens` on Claude).

**Exercise.** Write `hit_rate`: the share of **all** input tokens that were read from the cache. Each row
has `input_tokens`, and may have `cache_read` and `cache_write` (missing means 0). All input is the three
added together. With no input at all, return 0.

In [ ]:
def hit_rate(rows):
    """Share of all input tokens that were read from the cache."""
    # TODO: add up cache_read, and input_tokens + cache_read + cache_write, over the rows (use r.get(name, 0))
    raise NotImplementedError

In [ ]:
%%ipytest -qq


def test_one_write_then_one_read():
    rows = [{"input_tokens": 200, "cache_write": 1800}, {"input_tokens": 200, "cache_read": 1800}]
    assert hit_rate(rows) == pytest.approx(1800 / 4000)


def test_no_cache_fields_means_no_hits():
    assert hit_rate([{"input_tokens": 500}]) == 0.0


def test_nothing_sent():
    assert hit_rate([]) == 0.0

**When would it hit?** `t.cache_pass` replays a list of calls in order. A lesson's start is written on its
first call. Each later call within 5 minutes reads it. After a longer gap it's written again. Below, the
same dev questions are sent two ways: **grouped**, as a learner works through a lesson (a question about
every 40 seconds), and **spread out** over a day in random order. Each is tried with three minimums, to
show what decides the answer.

In [ ]:
def send_order(questions, grouped, seed=7):
    rng = random.Random(seed)
    if grouped:
        at, out, last = 0.0, [], None
        for q in questions:
            at += 40 if q["lesson_id"] == last else 600
            out.append({"lesson_id": q["lesson_id"], "key": q["key"], "at": at}); last = q["lesson_id"]
        return out
    return sorted(({"lesson_id": q["lesson_id"], "key": q["key"], "at": rng.uniform(0, 8 * 3600)} for q in questions), key=lambda c: c["at"])


start_of = dict(zip(starts.lesson_id, starts.cacheable))
total_of = {(q["lesson_id"], q["key"]): t.estimate_tokens(t.today_messages(DATA, q["lesson_id"], q["chip"])[0]["content"]) for q in QUESTIONS}
rows = []
for grouped in (True, False):
    order = send_order(QUESTIONS, grouped)
    sizes = [total_of[(c["lesson_id"], c["key"])] for c in order]
    for minimum in (512, 1024, 2048):
        cached = t.cache_pass(order, start_of, minimum=minimum)
        usage = [{"input_tokens": s - c["cache_read"] - c["cache_write"], **c} for s, c in zip(sizes, cached)]
        rows.append({"order": "grouped by lesson" if grouped else "spread over a day", "minimum": minimum, "hit_rate": round(hit_rate(usage), 3)})
pd.DataFrame(rows).pivot(index="order", columns="minimum", values="hit_rate")

**What to look for.** Two things decide the hit rate, and they're the two things to get right in any app.
**Length**: below the minimum, the hit rate is zero whatever you do. **Timing**: calls about the same
lesson must come close together. A learner working through a lesson does that. The same questions spread
over a day mostly miss.

A related trap: in a test run that sends every question in parallel, the cache only fills once the first
answer starts. Send one call first, then the rest.

## 3. The practice recording (made up, on real foundations)

> **In plain words.** Running seven versions live on a thousand questions costs real money (section 8
> estimates how much). So that you can learn the method for free, the cell below **simulates** one run.
> The made-up parts are clearly separate from the real ones. Swap in a real recording later and every
> cell after this one works the same.

| Part | Where it comes from |
|---|---|
| Questions, lessons, prompt lengths | **Real**: the dev questions and today's prompts |
| Prepared answers' quality and length | **Real**: each one's recorded judge verdict, and its text |
| Prices, cache rates, which calls hit the cache | **Real** rules (`tp.PRICES`, `t.cache_pass`), applied to the order below |
| Live models' pass rates, answer lengths and timings | **Made up** for practice: the numbers in `VERSIONS` |

The made-up quality isn't pure coin flips. Questions whose prepared answer failed are treated as
**harder** for every version, and some lessons are harder than others. Real versions behave like that:
they tend to fail on the same questions. That's why the comparison in section 5 pairs answers.

In [ ]:
SONNET, OPUS, HAIKU = "databricks-claude-sonnet-5-5", "databricks-claude-opus-5-5", "databricks-claude-haiku-5-5"
# name: (model, prompt layout, pass rate on an ordinary question, typical seconds to first words, words a second, answer length vs prepared)
VERSIONS = {  # MADE UP for practice, except "prepared, else Sonnet", whose prepared half is real
    "prepared, else Sonnet":   (SONNET, "today", None, None, None, None),
    "Opus":                    (OPUS, "today", .95, 2.4, 45, 1.00),
    "Sonnet":                  (SONNET, "today", .93, 1.5, 70, 0.95),
    "Sonnet, cacheable prompt": (SONNET, "cacheable", .93, 1.5, 70, 0.95),
    "Haiku":                   (HAIKU, "today", .86, 0.8, 120, 0.90),
    "Haiku, cacheable prompt": (HAIKU, "cacheable", .86, 0.8, 120, 0.90),
    "a local model":           ("local", "today", .74, 3.2, 20, 1.20),
}
BASELINE = "Sonnet"


def logit(p):
    return math.log(p / (1 - p))


def simulate(seed=7):
    rng = random.Random(seed)
    lesson_ease = {lid: rng.gauss(0, 0.6) for lid in sorted({q["lesson_id"] for q in QUESTIONS})}
    luck = {(q["lesson_id"], q["key"]): rng.random() for q in QUESTIONS}      # shared: versions fail on the same questions
    order = send_order(QUESTIONS, grouped=True)
    out = []

    def live(name, q, cache):
        model, layout, rate, ttft, speed, length = VERSIONS[name]
        hard = not q["prepared"]["judge"]["pass"]
        p = 1 / (1 + math.exp(-(logit(rate) + lesson_ease[q["lesson_id"]] - 1.6 * hard)))
        size = total_of[(q["lesson_id"], q["key"])]
        output = max(40, int(t.estimate_tokens(q["prepared"]["a"]) * length * rng.lognormvariate(0, 0.2)))
        read, write = (cache["cache_read"], cache["cache_write"]) if layout == "cacheable" else (0, 0)
        first = ttft * (1 - 0.3 * read / size) * rng.lognormvariate(0, 0.45) * (3 if rng.random() < 0.05 else 1)  # 1 in 20 waits behind a busy gateway
        return {"passed": luck[(q["lesson_id"], q["key"])] < p, "input_tokens": size - read - write, "cache_read": read, "cache_write": write,
                "output_tokens": output, "first_words_s": first, "total_s": first + output * 0.75 / speed,
                "cost": answer_cost(model, size - read - write, output, read, write), "model": model}

    for name, (model, layout, *_rest) in VERSIONS.items():
        caching = t.cache_pass(order, start_of if layout == "cacheable" else dict.fromkeys(start_of, 0))
        for q, cache in zip(QUESTIONS, caching):
            if name.startswith("prepared"):
                if q["prepared"]["judge"]["pass"]:   # shown at once, already paid for
                    row = {"passed": True, "input_tokens": 0, "cache_read": 0, "cache_write": 0, "output_tokens": 0,
                           "first_words_s": rng.uniform(0.03, 0.12), "total_s": 0.12, "cost": 0.0, "model": "prepared"}
                else:                                # a failed prepared answer is never shown: the live tutor answers instead
                    row = live("Sonnet", q, cache)
            else:
                row = live(name, q, cache)
            out.append({"version": name, "lesson_id": q["lesson_id"], "key": q["key"], **row})
    return pd.DataFrame(out)


REC = simulate()
REC["source"] = "practice"
print(f"{len(REC):,} answers: {REC.version.nunique()} versions x {len(QUESTIONS)} questions")
REC.head()

**What that cell did.** It produced one row per version per question, with the columns a real run
records: passed, input, cached and output tokens, seconds to first words, total seconds, cost. The
**prepared** version shows its answer if the judge passed it. If the judge failed it, the page never
shows it and asks the live tutor instead, here Sonnet, which pays for that answer. That's exactly what
the app does.

## 4. Speed: the slow case, not the average

> **In plain words.** Averages hide the slow answers, and those are the ones learners notice. So compare
> **p95**, the time 95 out of 100 answers arrive within (lesson "Latency that users feel"). For a chat,
> the time to the **first words** matters most, because once text starts appearing, people read along.

**Exercise.** Write `percentile`: sort the times, then take the one at position `ceil(n * p / 100)`,
counting from 1 (so subtract 1 for Python's index). Never go below position 1. An empty list gives 0.

In [ ]:
def percentile(times, p):
    """The time that p out of 100 requests finish within (nearest rank)."""
    # TODO
    raise NotImplementedError

In [ ]:
%%ipytest -qq


def test_median_of_ten():
    assert percentile([5, 1, 9, 3, 7, 2, 8, 4, 10, 6], 50) == 5


def test_p95_of_ten_is_the_slowest():
    assert percentile(list(range(1, 11)), 95) == 10


def test_one_time():
    assert percentile([3.0], 95) == 3.0


def test_no_times():
    assert percentile([], 95) == 0.0

In [ ]:
speed = REC.groupby("version", sort=False).agg(
    first_words_p50=("first_words_s", lambda s: percentile(list(s), 50)),
    first_words_p95=("first_words_s", lambda s: percentile(list(s), 95)),
    total_p95=("total_s", lambda s: percentile(list(s), 95)))
speed.round(2)

**What to look for.** The gap between the typical case (p50) and the slow one (p95). One answer in 20
here waits behind a busy gateway, which is what pulls the p95 up. Prepared answers are near zero most of
the time, but their p95 isn't, because the failed ones fall back to a live answer.

## 5. Quality: no worse than today, on the same questions

> **In plain words.** Every version answered the **same** questions, so compare them question by
> question. Only the questions where two versions differ tell you anything (lesson "Telling real gains
> from noise"). The question isn't "is it better?" but "is it **no worse** than today, by more than 3
> points?" That's a **non-inferiority** test. A cheaper version only needs to show it doesn't lose much.
>
> `t.paired_difference` gives the new pass rate minus the baseline's, with a 95% interval from
> resampling **whole lessons**. Answers about the same lesson aren't independent, so resampling single
> answers would make the interval look narrower than it really is. The rule passes a version when the
> **lower end** of the interval is above -3 points.

In [ ]:
verdicts = {v: {(r.lesson_id, r.key): bool(r.passed) for r in grp.itertuples()} for v, grp in REC.groupby("version", sort=False)}
quality = []
for v in VERSIONS:
    _, lo, hi = tpe.wilson(sum(verdicts[v].values()), len(verdicts[v]))
    diff, dlo, dhi = t.paired_difference(verdicts[BASELINE], verdicts[v])
    quality.append({"version": v, "pass_rate": sum(verdicts[v].values()) / len(verdicts[v]), "wilson_low": lo, "wilson_high": hi,
                    "vs_baseline": diff, "diff_low": dlo, "diff_high": dhi})
quality = pd.DataFrame(quality).set_index("version")
quality.round(3)

**What to look for.** The `diff_low` column. A version whose interval reaches well below -0.03 might be
worse than today by more than the margin, so it's out, however cheap it is. The baseline compared with
itself is exactly 0. Note the Wilson interval treats every answer as independent. It's printed only so
you can see it's narrower than the lesson-level one.

**Careful with the prepared version.** Its verdicts are real, but they come from the same judge that
decided which prepared answers to keep. So it's graded by the judge that chose it, and that flatters it.
In the real run, every version's answers, prepared ones included, are judged again by one judge. Make
that judge a different model from the one that did the choosing.

## 6. All three scores together, and the versions nothing beats

> **In plain words.** Put quality, cost and speed side by side. Then throw out every version that another
> version **beats**: at least as good on all three, and better on at least one. What's left is the
> **Pareto front**, the real trade-offs (lesson "Comparing LLMs for a task").

`t.beats(a, b)` says whether `a` beats `b`, using the `quality`, `cost` and `p95` of each.

**Exercise.** Write `pareto`: keep the name of each version that **no other** version beats. Compare
each version with every other one, but never with itself.

In [ ]:
def pareto(versions):
    """The names of the versions nothing beats."""
    # TODO: use t.beats(other, v)
    raise NotImplementedError

In [ ]:
%%ipytest -qq

VS = [{"name": "a", "quality": .9, "cost": 2, "p95": 3}, {"name": "b", "quality": .8, "cost": 1, "p95": 3},
      {"name": "c", "quality": .8, "cost": 3, "p95": 4}, {"name": "d", "quality": .9, "cost": 2, "p95": 3}]


def test_beaten_on_everything_is_dropped():
    assert "c" not in pareto(VS)


def test_each_wins_on_something():
    assert {"a", "b"} <= set(pareto(VS))


def test_a_tie_beats_neither():
    assert "a" in pareto(VS) and "d" in pareto(VS)

In [ ]:
table = pd.DataFrame({
    "quality": quality.pass_rate, "diff_low": quality.diff_low,
    "cost_per_1000": REC.groupby("version", sort=False).cost.mean() * 1000,
    "p95": speed.first_words_p95,
    "cache_hit_rate": pd.Series({v: hit_rate(g.to_dict("records")) for v, g in REC.groupby("version", sort=False)}),
})
versions = [{"name": v, "quality": r.quality, "cost": r.cost_per_1000, "p95": r.p95, "diff_low": r.diff_low} for v, r in table.iterrows()]
front = pareto(versions)
table["on_front"] = [v in front for v in table.index]
table.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
for v, r in table.iterrows():
    slow = r.p95 > P95_LIMIT
    ax.scatter(r.cost_per_1000, r.quality, s=80, marker="x" if slow else "o", alpha=1 if r.on_front else .35)
    below = "cacheable" in v  # its twin sits right next to it: label one above, one below
    ax.annotate(v + (" (too slow)" if slow else ""), (r.cost_per_1000, r.quality), textcoords="offset points",
                xytext=(6, -12 if below else 5), fontsize=8)
ax.axhline(table.loc[BASELINE, "quality"] - MARGIN, linestyle=":", linewidth=1)
ax.text(ax.get_xlim()[1], table.loc[BASELINE, "quality"] - MARGIN, "baseline - margin ", va="bottom", ha="right", fontsize=8)
ax.set_xlabel("dollars per 1,000 answers (lower is better)")
ax.set_ylabel("pass rate (higher is better)")
ax.set_title("Practice recording: faded = beaten by another version")
plt.show()

**What to look for.** Up and to the left is better. The faded points are beaten by something, so they
are never the right choice.

Look at the cacheable versions next to the ones they came from. About a third of their input comes from
the cache, but they're only a little cheaper. Input is the cheaper part of the bill, the first call on
each lesson pays a quarter extra to write the cache, and lessons with a short start get nothing. The
practice recording gives them the same quality as the originals, which is an assumption. In the real
run, check it: the model now reads the rules before the question.

The big move is **prepared, else Sonnet**. Most standard questions cost nothing to answer and come back
at once. Only the few prepared answers that failed their check pay for a live one.

## 7. Let the rule decide

**Exercise.** Write `choose`, the rule from section 0, in order:

1. drop every version whose `diff_low` is below `-margin` (reason `"quality"`);
2. of the rest, drop every version whose `p95` is above `p95_limit` (reason `"speed"`);
3. the choice is the cheapest version left (if two cost the same, the faster one), or `None` if none is left.

Return `{"choice": name or None, "kept": [names left, in order], "dropped": {name: reason}}`.

In [ ]:
def choose(versions, margin, p95_limit):
    """Quality first, then speed, then the cheapest of what's left."""
    # TODO
    raise NotImplementedError

In [ ]:
%%ipytest -qq

CANDIDATES = [{"name": "base", "diff_low": 0.0, "p95": 4, "cost": 5},
              {"name": "cheap-but-worse", "diff_low": -0.08, "p95": 2, "cost": 1},
              {"name": "cheap-and-slow", "diff_low": -0.01, "p95": 9, "cost": 1},
              {"name": "cheaper", "diff_low": -0.02, "p95": 3, "cost": 2}]


def test_the_cheapest_that_holds_quality_and_speed():
    assert choose(CANDIDATES, margin=0.03, p95_limit=5)["choice"] == "cheaper"


def test_it_says_why_each_was_dropped():
    assert choose(CANDIDATES, margin=0.03, p95_limit=5)["dropped"] == {"cheap-but-worse": "quality", "cheap-and-slow": "speed"}


def test_nothing_left():
    assert choose(CANDIDATES[1:3], margin=0.03, p95_limit=5)["choice"] is None

In [ ]:
decision = choose(versions, MARGIN, P95_LIMIT)
print("kept:   ", decision["kept"])
print("dropped:", decision["dropped"])
print("choice: ", decision["choice"], " <- from the practice recording: the method, not the answer")

**What to look for.** The rule says *why* each version lost, which is the part to show a manager. Note
what it doesn't do: it never looks at the chart and picks a favourite. If you disagree with the choice,
change the plan in section 0 and say so. Don't change the result.

To see what the money is worth, the course's lesson "Net value per month" adds the other side: the value
of a good answer. A version that costs more can still be the better buy if it helps more learners.

## 8. Before the real run: how many questions, and what it costs

> **In plain words.** A real run sends every question to every version, and then a judge grades every
> answer. Both cost money. So first work out how many questions you need to tell a 3-point drop from
> luck (lesson "How many examples do you need?"), then what that many will cost.

For a paired non-inferiority test, a common rule of thumb is: questions ≈ (1.645 + 0.84)² ×
(share of questions where the two versions differ) ÷ margin². That's 95% sure, with an 80% chance of
spotting a real drop. Answers come in lessons, so add about half again: lesson-level intervals are wider,
as the judging notebook's section 6 shows.

The rule assumes the two versions are really equal, which is the hardest case to call. A version that is
clearly better or worse shows it with far fewer questions. So size the run for the **close calls**,
the versions within a few points of the baseline.

In [ ]:
differ = {v: sum(verdicts[v][k] != verdicts[BASELINE][k] for k in verdicts[BASELINE]) / len(verdicts[BASELINE]) for v in VERSIONS if v != BASELINE}
needed = {v: math.ceil((1.645 + 0.84) ** 2 * d / MARGIN ** 2 * 1.5) for v, d in differ.items()}
close = [v for v in needed if abs(quality.loc[v, "vs_baseline"]) <= 2 * MARGIN and needed[v]]
n = min(max([needed[v] for v in close] or [len(QUESTIONS)]), len(QUESTIONS))
print("questions needed per version:", needed)
print("close calls:", close, "(0 needed means the practice recording made two versions answer alike)")
print(f"-> run {n} questions (the dev set has {len(QUESTIONS)})")

# Judging, from the real drafting run: Sonnet judged 2,094 answers for 3,429,008 input and 2,212,630 output tokens
judge_each = tp.cost(SONNET, 3_429_008, 2_212_630) / 2094
answers_each = REC.groupby("version", sort=False).cost.mean()
run = pd.DataFrame({"answers": answers_each * n, "judging": judge_each * n})
run.loc["total"] = run.sum()
run.round(2)

**What to look for.** The total in the last row, and which part drives it. Judging every answer with a
strong model often costs more than the answers themselves. Two ways to spend less: judge with a cheaper
model, once you've checked you can trust it (the judging notebook), or drop versions the plan already
rules out. The real runner prints this estimate and asks before it spends anything.

## 9. Track it in MLflow

> **In plain words.** One **run** per version, all inside one parent run for the experiment. Each run
> logs its settings (params), its scores (metrics) and a `source` tag, so a practice run can never be
> mistaken for a real one. The parent run holds the plan, the decision and the chart. This uses a
> **throwaway** store in a temporary folder, so it costs nothing.

In [ ]:
import os
import tempfile

os.environ.setdefault("MLFLOW_DISABLE_TELEMETRY", "true")
import mlflow

mlflow.set_tracking_uri(f"sqlite:///{tempfile.mkdtemp(prefix='tradeoff-mlflow-')}/mlflow.db")
mlflow.set_experiment("tutor-cost-speed-quality-learning")
with mlflow.start_run(run_name="tradeoff-dev-practice") as parent:
    mlflow.set_tags({"split": "dev", "source": "practice", "purpose": "learning", "decision": str(decision["choice"])})
    mlflow.log_params({"baseline": BASELINE, "margin": MARGIN, "p95_limit_s": P95_LIMIT, "questions": len(QUESTIONS),
                       "min_cacheable": t.MIN_CACHEABLE, "decision_rule": PLAN["decision_rule"]})
    mlflow.log_dict(PLAN, "plan.json")
    mlflow.log_dict(decision, "decision.json")
    mlflow.log_figure(fig, "front.png")
    for v, r in table.iterrows():
        model, layout, *_ = VERSIONS[v]
        with mlflow.start_run(run_name=f"version-{v.lower().replace(',', '').replace(' ', '-')}", nested=True):
            mlflow.set_tags({"split": "dev", "source": "practice", "on_front": str(r.on_front)})
            mlflow.log_params({"model": model, "prompt": layout})
            mlflow.log_metrics({"pass_rate": r.quality, "diff_low": r.diff_low, "cost_per_1000_usd": r.cost_per_1000,
                                "first_words_p95_s": r.p95, "cache_hit_rate": r.cache_hit_rate})
found = mlflow.search_runs(filter_string=f"tags.mlflow.parentRunId = '{parent.info.run_id}'")
found[["tags.mlflow.runName", "params.model", "params.prompt", "metrics.pass_rate", "metrics.cost_per_1000_usd", "metrics.first_words_p95_s"]].round(3)

## 10. Confirm once, on test

Everything above is **dev**: try as many plans and versions as you like. When the rule has chosen, run
**only the winner and the baseline** on the **test** lessons, once, and report that number. Holdout stays
for a bigger decision, such as switching the live tutor to a different model (the judging notebook,
section 16, shows how to make MLflow enforce this).

The real version of this notebook is one command that runs the versions live, on a Databricks profile you
choose, after printing the estimate from section 8. It writes a recording with the same columns as `REC`,
so you can load it in place of `simulate()` and run every cell again.

## What you used

| Idea | Course lesson | In this notebook |
|---|---|---|
| Write the rule first | Write the plan before the runs | section 0 |
| Tokens and cost | Tokens and cost | section 1 |
| Cache hit rate | Caching responses | section 2 |
| p95, not the average | Latency that users feel | section 4 |
| Paired comparison | Telling real gains from noise | section 5 |
| The Pareto front | Comparing LLMs for a task | section 6 |
| What the money is worth | Net value per month | section 7 |
| How many questions | How many examples do you need? | section 8 |

Beyond the course: the provider's prompt cache (its minimum length, its few-minute lifetime, and why
order and timing decide the hit rate), non-inferiority with a lesson-level paired interval, and a
practice recording built on real foundations.